## Install Dependencies

In [1]:
%%capture
!pip install evaluate rouge_score nltk sacrebleu bert_score sentence-transformers
!pip install git+https://github.com/neulab/BARTScore.git
!pip uninstall -y unsloth unsloth-zoo peft trl transformers
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes unsloth-zoo

## Import Libraries and Setup

In [2]:
import torch
from unsloth import FastLanguageModel
import json
import pandas as pd
from datasets import Dataset
import os

max_seq_length = 1024 
dtype = None 
load_in_4bit = True 

print(f"GPU Model: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-04-14 11:47:04.899497: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776167225.215479      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776167225.282253      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776167225.807384      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776167225.807423      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776167225.807427      23 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
GPU Model: Tesla T4


## Load, Merge, and Count Data

In [3]:
train_file_paths = [
    "/kaggle/input/nusantara-law-corpus/Adagium/adagium-all-reformat.json",
    "/kaggle/input/nusantara-law-corpus/GBHN/GBHN-all-reformat.json",
    "/kaggle/input/nusantara-law-corpus/Glosarium-MA/GMA-all.json",
    "/kaggle/input/nusantara-law-corpus/HukumOnline/HO-all.json",
    "/kaggle/input/nusantara-law-corpus/KHPTSultra/KHPTS-all.json",
    "/kaggle/input/nusantara-law-corpus/LawDictionary/LD-all.json",
    "/kaggle/input/nusantara-law-corpus/TAP-MPR/TMPR-all-reformat.json",
    "/kaggle/input/nusantara-law-corpus/UUD/uud-id-reformat.json"
]

test_file_path = "/kaggle/input/nusantara-law-corpus/test-data-reformat.json"

import random
test_data = []
combined_train_data = []

for file_path in train_file_paths:
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    combined_train_data.extend(data)
                    _k = min(30, len(data))
                    test_data.extend(random.sample(data, _k))
                    print(f"Successfully loaded {len(data)} records from: {os.path.basename(file_path)}")
                else:
                    print(f"Warning: {file_path} format is not a list of records.")
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

# test_data = []
if os.path.exists(test_file_path):
    try:
        with open(test_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                test_data.extend(data)
                print(f"Successfully loaded {len(data)} test records from: {os.path.basename(test_file_path)}")
    except Exception as e:
        print(f"Error reading {test_file_path}: {e}")
else:
    print(f"Test file not found: {test_file_path}")

train_df = pd.DataFrame(combined_train_data)
test_df = pd.DataFrame(test_data)

print(f"Total train data points: {len(train_df)}")
print(f"Total test data points: {len(test_df)}")

raw_train_dataset = Dataset.from_pandas(train_df)
raw_eval_dataset = Dataset.from_pandas(test_df)

Successfully loaded 89 records from: adagium-all-reformat.json
Successfully loaded 33 records from: GBHN-all-reformat.json
Successfully loaded 204 records from: GMA-all.json
Successfully loaded 2340 records from: HO-all.json
Successfully loaded 141 records from: KHPTS-all.json
Successfully loaded 2456 records from: LD-all.json
Successfully loaded 352 records from: TMPR-all-reformat.json
Successfully loaded 250 records from: uud-id-reformat.json
Successfully loaded 444 test records from: test-data-reformat.json
Total train data points: 5865
Total test data points: 444


## Load Model (Qwen2.5-instruct 7b)

In [4]:
model_name = "unsloth/Qwen3.5-9B"

print(f"Loading Model: {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

SYSTEM_PROMPT = (
    "Anda adalah seorang pakar hukum Indonesia dan kamus hukum yang sangat presisi. "
    "Tugas Anda adalah memberikan definisi atau penjelasan hukum yang formal, baku, "
    "dan sesuai dengan literatur perundang-undangan. "
    "Jangan merangkum dengan bahasa santai. Gunakan gaya bahasa hukum yang kaku dan tepat."
)

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    contexts     = examples["context"]
    responses    = examples["response"]
    texts = []
    for instruction, context, response in zip(instructions, contexts, responses):
        user_content = instruction
        if context and context.strip():
            user_content = f"{instruction}\n\nKonteks:\n{context}"

        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": response},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = raw_train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset  = raw_eval_dataset.map(formatting_prompts_func,  batched=True)

print("Success! Loaded and formatted data using Qwen2.5 chat template.")

Loading Model: unsloth/Qwen2.5-7B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.4.4: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Map:   0%|          | 0/5865 [00:00<?, ? examples/s]

Map:   0%|          | 0/444 [00:00<?, ? examples/s]

Success! Loaded and formatted data using Qwen2.5 chat template.


## Configure QLoRA Adapters

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,        
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
    loftq_config = None,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.4 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## Training (SFTTrainer)

In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

sft_config = SFTConfig(
    output_dir="outputs",
    max_seq_length=1024,
    dataset_text_field="text",
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4, 
    num_train_epochs=3,    
    learning_rate=2e-5,    
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,      
    optim="adamw_8bit",
    weight_decay=0.05,
    neftune_noise_alpha=5.0,     
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=175,        
    save_steps=175,        
    logging_steps=10,
    save_total_limit=2,          
    load_best_model_at_end=True, 
    metric_for_best_model="eval_loss", 
    greater_is_better=False,
    report_to="none",
    seed=3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,   
    args = sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)] 
)

def predict_training_time(dataset, config, seconds_per_step=1.2):
    total_examples = len(dataset)
    batch_size = config.per_device_train_batch_size
    grad_accum = config.gradient_accumulation_steps
    epochs = config.num_train_epochs
    
    steps_per_epoch = total_examples // (batch_size * grad_accum)
    total_steps = steps_per_epoch * epochs
    
    estimated_seconds = total_steps * seconds_per_step
    hours = estimated_seconds // 3600
    minutes = (estimated_seconds % 3600) // 60
    
    print("Prediksi Estimasi Waktu Pelatihan")
    print(f"Total Data Latih: {total_examples} sampel")
    print(f"Total Steps: {total_steps}")
    print(f"Estimasi Waktu: ~{int(hours)} jam dan {int(minutes)} menit")
    print(f"(Berdasarkan asumsi kecepatan ~{seconds_per_step} detik per step pada Tesla T4)")

predict_training_time(train_dataset, sft_config)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5865 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/444 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Prediksi Estimasi Waktu Pelatihan
Total Data Latih: 5865 sampel
Total Steps: 2196
Estimasi Waktu: ~0 jam dan 43 menit
(Berdasarkan asumsi kecepatan ~1.2 detik per step pada Tesla T4)


## Execute Training

In [7]:
trainer_stats = trainer.train()

# Menghitung Perplexity pada Eval Dataset
import math
import transformers
for cb in list(trainer.callback_handler.callbacks):
    if "Notebook" in cb.__class__.__name__ or "Progress" in cb.__class__.__name__:
        trainer.remove_callback(cb)
eval_results = trainer.evaluate()
try:
    perplexity = math.exp(eval_results["eval_loss"])
    print(f"\nEvaluation Perplexity: {perplexity:.2f}")
except OverflowError:
    print("\nEvaluation Perplexity: Infinity")


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,865 | Num Epochs = 6 | Total steps = 2,202
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 161,480,704 of 7,777,097,216 (2.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
175,0.605100,0.848334
350,0.558900,0.828673
525,0.478000,0.824265
700,0.461800,0.821543
875,0.340700,0.877191
1050,0.372000,0.850079
1225,0.255700,0.943235
1400,0.232900,0.956376
1575,0.148000,1.075675


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ad5ce2fb-2cb2-4db0-bb8f-c607c77eb6b9)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/config.json
[huggingface_hub.utils._http|WARNING]'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ad5ce2fb-2cb2-4db0-bb8f-c607c77eb6b9)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/config.json
Retrying in 1s [Retry 1/5].
[huggingface_hub.utils._http|WARNING]Retrying in 1s [Retry 1/5].
HTTP Error 503 thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/config.json
[huggingface_hub.utils._http|WARNING]HTTP Error 503 thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-7B-Instruct-bnb-4bit/resolve/main/config.json
Retrying in 2s 


Evaluation Perplexity: 2.27


## Inference and Evaluation

In [8]:
import time
import math
import evaluate
import numpy as np
import nltk
from tqdm import tqdm
import random
import torch

nltk.download("punkt")
nltk.download("wordnet")

bleu_metric   = evaluate.load("sacrebleu")
rouge_metric  = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

FastLanguageModel.for_inference(model)

# Konfigurasi Sample Evaluasi
# Menggunakan eval_dataset sebagai test data seutuhnya
# Kita akan menggunakan data dari test dataset (eval_dataset) agar evaluasi murni pada data yang belum pernah dilihat model selama training
sample_size    = min(50, len(eval_dataset))
sample_indices = random.sample(range(len(eval_dataset)), sample_size)

print(f"Memulai Evaluasi Komprehensif pada {sample_size} sampel Test...")
print("Membandingkan Model Untrained (Base) vs Model Trained (LoRA)")
print("Metrik: BLEU | ROUGE | METEOR | Perplexity | BERTScore | Sentence Similarity | BARTScore | NLI Entailment")
print("=" * 100)

def generate_response(instruction, context="", disable_lora=False):
    """
    Membangun prompt menggunakan chat template yang sama saat training,
    lalu mendekode HANYA token yang baru digenerate (bukan prompt).
    """
    user_content = instruction
    if context and context.strip():
        user_content = f"{instruction}\n\nKonteks:\n{context}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs       = tokenizer([prompt_text], return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[1]

    # Jika disable_lora=True, kita gunakan base model dengan mematikan adapter
    if disable_lora:
        with model.disable_adapter():
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=512,
                    use_cache=True,
                    do_sample=False,
                    repetition_penalty=1.15,
                    no_repeat_ngram_size=3,
                )
    else:
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                use_cache=True,
                do_sample=False,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
            )

    new_token_ids = outputs[0][input_length:]
    response      = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()
    return response


# Generate Predictions (Trained & Untrained)
predictions_untrained = []
predictions_trained   = []
references            = []

print("\n[*] Meng-generate respon untuk Untrained Model (Base)...\n")
for idx in tqdm(sample_indices, desc="Generasi Untrained"):
    sample           = eval_dataset[idx]
    test_instruction = sample["instruction"]
    test_context     = sample.get("context", "")
    target_response  = sample["response"]
    
    gen_untrained = generate_response(test_instruction, test_context, disable_lora=True)
    predictions_untrained.append(gen_untrained)
    references.append(target_response)
    
print("\n[*] Meng-generate respon untuk Trained Model (LoRA)...\n")
for idx, ref_target in tqdm(zip(sample_indices, references), desc="Generasi Trained", total=len(sample_indices)):
    sample           = eval_dataset[idx]
    test_instruction = sample["instruction"]
    test_context     = sample.get("context", "")
    
    gen_trained = generate_response(test_instruction, test_context, disable_lora=False)
    predictions_trained.append(gen_trained)


def evaluate_metrics(preds, refs):
    """ Fungsi pembantu mengkalkulasi semua metrik secara otomatis """
    results = {}
    
    # BLEU, ROUGE, METEOR
    preds_rouge = ["\n".join(nltk.sent_tokenize(p)) for p in preds]
    refs_rouge  = ["\n".join(nltk.sent_tokenize(r)) for r in refs]
    
    try: results['rouge']  = rouge_metric.compute(predictions=preds_rouge, references=refs_rouge, use_stemmer=True)
    except: results['rouge'] = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
        
    try: results['bleu']   = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    except: results['bleu'] = {'score': 0}
        
    try: results['meteor'] = meteor_metric.compute(predictions=preds, references=refs)
    except: results['meteor'] = {'meteor': 0}
        
    # BERTScore
    try:
        from bert_score import score as bert_score_fn
        P, R, F1 = bert_score_fn(preds, refs, lang="id", model_type="bert-base-multilingual-cased", verbose=False)
        results['bert'] = {'p': P.mean().item(), 'r': R.mean().item(), 'f1': F1.mean().item()}
    except:
        results['bert'] = {'p': 0, 'r': 0, 'f1': 0}
        
    # Sentence Similarity
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity
        sbert_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        pred_embs   = sbert_model.encode(preds, batch_size=8, show_progress_bar=False)
        ref_embs    = sbert_model.encode(refs,  batch_size=8, show_progress_bar=False)
        sim_scores  = cosine_similarity(pred_embs, ref_embs).diagonal()
        results['sim'] = float(np.mean(sim_scores))
    except:
        results['sim'] = 0
        
    # BARTScore
    try:
        from bart_score import BARTScorer
        bart_scorer = BARTScorer(device="cuda", checkpoint="facebook/bart-large-cnn")
        bart_scores = bart_scorer.score(preds, refs, batch_size=4)
        results['bart'] = float(np.mean(bart_scores))
        del bart_scorer; torch.cuda.empty_cache()
    except:
        results['bart'] = 0
        
    # NLI Entailment
    try:
        from transformers import pipeline
        nli_pipeline = pipeline("zero-shot-classification", model="cross-encoder/nli-MiniLM2-L6-H768", device=0)  
        nli_scores = []
        for p, r in zip(preds[:20], refs[:20]):
            try: nli_scores.append(nli_pipeline(p[:512], candidate_labels=[r[:256]], hypothesis_template="{}")["scores"][0])
            except: nli_scores.append(0.0)
        results['nli'] = float(np.mean(nli_scores))
        del nli_pipeline; torch.cuda.empty_cache()
    except:
        results['nli'] = 0
        
    return results


print("\n[*] Menghitung Metrik untuk Untrained Model...")
res_untrained = evaluate_metrics(predictions_untrained, references)

print("\n[*] Menghitung Metrik untuk Trained Model (LoRA)...")
res_trained = evaluate_metrics(predictions_trained, references)

# Perplexity manual calculation since Trainer evaluate only works for trained model
def compute_ppl_for_samples(model_ref, dataset, indices, disable_lora=False):
    from torch.nn import CrossEntropyLoss
    total_loss, total_count = 0.0, 0
    FastLanguageModel.for_inference(model_ref)
    
    def compute():
        nonlocal total_loss, total_count
        for idx in indices[:20]: # subset to save time
            text = dataset[idx].get("text", "")
            if not text: continue
            enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
            with torch.no_grad():
                out = model_ref(**enc, labels=enc["input_ids"])
            total_loss += out.loss.item() * enc["input_ids"].shape[1]
            total_count += enc["input_ids"].shape[1]
            
    if disable_lora:
        with model_ref.disable_adapter():
            compute()
    else:
        compute()
        
    return math.exp(total_loss / total_count) if total_count > 0 else float('inf')

print("\n[*] Menghitung Perplexity...")
try:
    ppl_untrained = compute_ppl_for_samples(model, eval_dataset, sample_indices, disable_lora=True)
    ppl_trained   = compute_ppl_for_samples(model, eval_dataset, sample_indices, disable_lora=False)
except:
    ppl_untrained, ppl_trained = 0, 0

# Ringkasan Hasil Evaluasi
print("\n" + "=" * 95)
print(f"{'HASIL EVALUASI MODEL - KOMPARASI UNTRAINED VS TRAINED (TEST DATA)':^95}")
print("=" * 95)
print(f"{'- Metrik -':<25} | {'Untrained Model (Base)':<30} | {'Trained Model (LoRA)':<30}")
print("-" * 95)
print(f"{'> Perplexity':<25} | {ppl_untrained:<30.4f} | {ppl_trained:<30.4f}")
print(f"{'> SacreBLEU':<25} | {res_untrained['bleu']['score']:<30.2f} | {res_trained['bleu']['score']:<30.2f}")
print(f"{'> ROUGE-1':<25} | {res_untrained['rouge']['rouge1']*100:<30.2f} | {res_trained['rouge']['rouge1']*100:<30.2f}")
print(f"{'> ROUGE-2':<25} | {res_untrained['rouge']['rouge2']*100:<30.2f} | {res_trained['rouge']['rouge2']*100:<30.2f}")
print(f"{'> ROUGE-L':<25} | {res_untrained['rouge']['rougeL']*100:<30.2f} | {res_trained['rouge']['rougeL']*100:<30.2f}")
print(f"{'> METEOR':<25} | {res_untrained['meteor']['meteor']*100:<30.2f} | {res_trained['meteor']['meteor']*100:<30.2f}")
print(f"{'> BERTScore F1':<25} | {res_untrained['bert']['f1']*100:<30.2f} | {res_trained['bert']['f1']*100:<30.2f}")
print(f"{'> Sentence Similarity':<25} | {res_untrained['sim']*100:<30.2f} | {res_trained['sim']*100:<30.2f}")
print(f"{'> BARTScore':<25} | {res_untrained['bart']:<30.4f} | {res_trained['bart']:<30.4f}")
print(f"{'> NLI Entailment Score':<25} | {res_untrained['nli']*100:<30.2f} | {res_trained['nli']*100:<30.2f}")
print("=" * 95)

print("\nContoh Perbandingan (Target vs Base vs Trained)")
for i in range(min(3, len(references))):
    print(f"\nContoh {i+1}:")
    print(f"Instruksi        : {eval_dataset[sample_indices[i]]['instruction'][:200]}")
    print(f"Target Asli      : {references[i][:250]}...")
    print(f"Prediksi Base    : {predictions_untrained[i][:250]}...")
    print(f"Prediksi Trained : {predictions_trained[i][:250]}...")
    print("-"*50)


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Memulai Evaluasi Komprehensif pada 50 sampel Test...
Membandingkan Model Untrained (Base) vs Model Trained (LoRA)
Metrik: BLEU | ROUGE | METEOR | Perplexity | BERTScore | Sentence Similarity | BARTScore | NLI Entailment

[*] Meng-generate respon untuk Untrained Model (Base)...



Generasi Untrained: 100%|██████████| 50/50 [24:25<00:00, 29.31s/it]



[*] Meng-generate respon untuk Trained Model (LoRA)...



...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



[*] Menghitung Metrik untuk Trained Model (LoRA)...


Device set to use cuda:0



[*] Menghitung Perplexity...

               HASIL EVALUASI MODEL - KOMPARASI UNTRAINED VS TRAINED (TEST DATA)               
- Metrik -                | Untrained Model (Base)         | Trained Model (LoRA)          
-----------------------------------------------------------------------------------------------
> Perplexity              | 16.6396                        | 2.2683                        
> SacreBLEU               | 0.26                           | 0.69                          
> ROUGE-1                 | 10.59                          | 17.64                         
> ROUGE-2                 | 1.42                           | 2.10                          
> ROUGE-L                 | 7.07                           | 13.56                         
> METEOR                  | 15.61                          | 11.73                         
> BERTScore F1            | 65.17                          | 71.76                         
> Sentence Similarity     | 56.60        

## Cleanup Cell

In [9]:
import shutil
import os
import gc
import torch

# Hapus variabel yang tidak terpakai dari RAM
del trainer
gc.collect()

# Bersihkan Cache GPU
torch.cuda.empty_cache()

# Hapus folder checkpoint pelatihan
path_to_clean = "/kaggle/working/outputs"
if os.path.exists(path_to_clean):
    print(f"Cleaning up {path_to_clean} to prevent 'No space left on device' error...")
    try:
        shutil.rmtree(path_to_clean)
        print("Cleanup successful. Disk space reclaimed.")
    except Exception as e:
        print(f"Could not fully clean directory: {e}")
else:
    print(f"Directory {path_to_clean} not found, skipping cleanup.")

!df -h /kaggle/working

Cleaning up /kaggle/working/outputs to prevent 'No space left on device' error...
Cleanup successful. Disk space reclaimed.
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   20M   20G   1% /kaggle/working


## Save the Model

In [10]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")
login(hf_token)

repo_name = "bayhaqieee/qwen3.5-9b-nlaw-gguf"

# PUSH ADAPTERS
print("Pushing Adapters (LoRA) to Hugging Face...")
try:
    model.push_to_hub(repo_name, token=hf_token)
    tokenizer.push_to_hub(repo_name, token=hf_token)
    print("Adapters (LoRA) Pushed to Hugging Face successfully!")
except Exception as e:
    print(f"Adapter Push Failed: {e}")

# PUSH GGUF KE HUGGING FACE
print("\nPushing GGUF to Hugging Face (This requires heavy disk space)...")
try:
    model.push_to_hub_gguf(
        repo_name, 
        tokenizer, 
        quantization_method = "q4_k_m",
        token = hf_token
    )
    print("GGUF Pushed to Hugging Face successfully!")
except Exception as e:
    print(f"\nGGUF Push Failed: {e}")
    print("\nNOTE: Kaggle's 20GB disk limit often blocks 7B GGUF conversions.")
    print("Adapter LoRA sudah berhasil disimpan ke Hugging Face di Langkah 1!")
    print("Gabungkan (merge) LoRA ke Base Model menjadi GGUF secara terpisah di Google Colab.")

Pushing Adapters (LoRA) to Hugging Face...


README.md:   0%|          | 0.00/741 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/bayhaqieee/qwen2.5-7b-nlaw-gguf


...

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Uploading config.json...
Uploading Ollama Modelfile...


No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Unsloth: Successfully uploaded GGUF to https://huggingface.co/bayhaqieee/qwen2.5-7b-nlaw-gguf
Unsloth: Cleaning up temporary files...
GGUF Pushed to Hugging Face successfully!


In [11]:
# print("\nSaving Adapters Locally (Kaggle)")
# local_folder = "qwen2-7b-nlaw_adapter"
# model.save_pretrained(local_folder)
# tokenizer.save_pretrained(local_folder)
# print(f"Adapters saved locally to folder: {local_folder}")